# Module 1 Assignment Project

Designing a simple data product (dashboard) using real Singapore Job's Posting csv data. 

Our team is addressing a gap faced by companies entering the Singapore market: no reliable, industry-specific benchmark for talent costs or local hiring pool depth, even though headcount is typically the largest line item in an entry budget.

Our goal is to establish insights for clients, so that they can select their target industry and receive an evidence-based view of prevailing salary ranges by industry or by job title roles with data from 2023 - 2024.

### Structure

- Load and inspect initial EDA
- Remove missing values
- Clean date formats
- Clean title and text
- Remove duplicate postings
- Validate positions levels and experience
- Clean and flag unreliable salaries
- Convert hourly and annual salaries to monthly
- Parse the categories JSON
- .explode() into the long category table
- Job roles filtering from 'title_clean'
- Build analysis groupings
- Write the cleaned files
- Build the Streamlit dashboard

> The one design decision that matters is to acknowledge categories is one-to-many: a posting can sit in several industries. We keep a wide table at one row per posting for all headline numbers, and a separate exploded table at one row per posting x category for industry breakdowns.

In [1]:
import pandas as pd
import numpy as np
import re
import json

## 1. Exploratory Data Analysis

In [3]:
df = pd.read_csv('SGJobData.csv')

#Initial exploratory info

df.info()
df.describe(include='all')


<class 'pandas.DataFrame'>
RangeIndex: 1048585 entries, 0 to 1048584
Data columns (total 22 columns):
 #   Column                              Non-Null Count    Dtype  
---  ------                              --------------    -----  
 0   categories                          1044597 non-null  str    
 1   employmentTypes                     1044597 non-null  str    
 2   metadata_expiryDate                 1044597 non-null  str    
 3   metadata_isPostedOnBehalf           1048585 non-null  bool   
 4   metadata_jobPostId                  1044597 non-null  str    
 5   metadata_newPostingDate             1044597 non-null  str    
 6   metadata_originalPostingDate        1044597 non-null  str    
 7   metadata_repostCount                1048585 non-null  int64  
 8   metadata_totalNumberJobApplication  1048585 non-null  int64  
 9   metadata_totalNumberOfView          1048585 non-null  int64  
 10  minimumYearsExperience              1048585 non-null  int64  
 11  numberOfVacancies     

,categories,employmentTypes,metadata_expiryDate,metadata_isPostedOnBehalf,metadata_jobPostId,metadata_newPostingDate,metadata_originalPostingDate,metadata_repostCount,metadata_totalNumberJobApplication,metadata_totalNumberOfView,...,occupationId,positionLevels,postedCompany_name,salary_maximum,salary_minimum,salary_type,status_id,status_jobStatus,title,average_salary
count,1044597,1044597,1044597,1048585,1044597,1044597,1044597,1.048585e+06,1.048585e+06,1.048585e+06,...,0.0,1044597,1044597,1.048585e+06,1.048585e+06,1044597,1048585.0,1044597,1044597,1.048585e+06
unique,21125,8,453,2,1044597,431,603,NaN,NaN,NaN,...,NaN,9,53151,NaN,NaN,1,NaN,3,377084,NaN
top,"[{""id"":21,""category"":""Information Technology""}]",Permanent,2023-07-28,False,MCF-2023-0252866,2023-06-09,2023-07-14,NaN,NaN,NaN,...,NaN,Executive,THE SUPREME HR ADVISORY PTE. LTD.,NaN,NaN,Monthly,NaN,Open,SUPERVISOR,NaN
freq,92869,458139,4487,986717,1,4508,4029,NaN,NaN,NaN,...,NaN,253701,61638,NaN,NaN,1044597,NaN,902614,8331,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.472327e-02,2.136571e+00,2.674536e+01,...,NaN,NaN,NaN,5.723578e+03,3.815312e+03,NaN,0.0,NaN,NaN,4.769445e+03
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.822675e-01,1.062612e+01,8.262001e+01,...,NaN,NaN,NaN,5.018387e+04,3.172182e+03,NaN,0.0,NaN,NaN,2.547809e+04
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,0.000000e+00,...,NaN,NaN,NaN,0.000000e+00,0.000000e+00,NaN,0.0,NaN,NaN,0.000000e+00
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,1.000000e+00,...,NaN,NaN,NaN,3.300000e+03,2.500000e+03,NaN,0.0,NaN,NaN,2.900000e+03
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,4.000000e+00,...,NaN,NaN,NaN,4.500000e+03,3.000000e+03,NaN,0.0,NaN,NaN,3.800000e+03
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,1.000000e+00,1.700000e+01,...,NaN,NaN,NaN,6.500000e+03,4.500000e+03,NaN,0.0,NaN,NaN,5.500000e+03


### Initial information gathered

- Total of **1048585 rows** and **22 columns**
- Some have missing rows (showing 1044597 only): **3988 missing values**
- Column **'occupationID'** is totally empty (all Null values)
- **Dated columns listed as object** --> Indicating string format (need to convert)
- Column **'categories' listed as object** (looks like json, but in string format; to check)
- 'salary_minimum' and 'salary_maximum' have **outliers**

## 1.1 Removing Missing Values (NaN)

In [4]:
# Checking missing values

missing_values = df.isna().sum()

print(missing_values[missing_values > 0].sort_values(ascending=False))

occupationId                    1048585
categories                         3988
employmentTypes                    3988
metadata_expiryDate                3988
metadata_jobPostId                 3988
metadata_newPostingDate            3988
metadata_originalPostingDate       3988
positionLevels                     3988
postedCompany_name                 3988
salary_type                        3988
status_jobStatus                   3988
title                              3988
dtype: int64


Similar number of missing values. To drop these rows as it does not help in analysis (no usable data value).

In [5]:
# Entire row of occupationId is empty. Initial df.dropna() via rows will drop every single row. Hence drop via column.

# Drop empty occupationId column
df_clean = df.drop(columns=['occupationId'])

# We use jobs id column which is unique, except for the 3988 empty rows, to drop the empty jobs id rows.
df_clean = df_clean.dropna(subset=['metadata_jobPostId']).copy()

#check if rows are dropped
df_clean.isna().sum()

categories                            0
employmentTypes                       0
metadata_expiryDate                   0
metadata_isPostedOnBehalf             0
metadata_jobPostId                    0
metadata_newPostingDate               0
metadata_originalPostingDate          0
metadata_repostCount                  0
metadata_totalNumberJobApplication    0
metadata_totalNumberOfView            0
minimumYearsExperience                0
numberOfVacancies                     0
positionLevels                        0
postedCompany_name                    0
salary_maximum                        0
salary_minimum                        0
salary_type                           0
status_id                             0
status_jobStatus                      0
title                                 0
average_salary                        0
dtype: int64

## 1.2 Parsing Date type

In [6]:
# Parsing the date columns - convert date object to datetime 

date_columns = ['metadata_expiryDate', 'metadata_newPostingDate', 'metadata_originalPostingDate']

for c in date_columns:
    df_clean[c] = pd.to_datetime(df_clean[c], errors='coerce')

In [7]:
print('Check for unparseable dates:', {c: df_clean[c].isna().sum() for c in date_columns})
print('Data date range:', df_clean['metadata_originalPostingDate'].min().date(), '->',
                     df_clean['metadata_originalPostingDate'].max().date())

Check for unparseable dates: {'metadata_expiryDate': np.int64(0), 'metadata_newPostingDate': np.int64(0), 'metadata_originalPostingDate': np.int64(0)}
Data date range: 2022-10-03 -> 2024-05-29


## 1.3 Cleaning 'title' Column

'title' column was extremely messy with various forms of input since it is free text. To avoid over-normalizing the data,
Removing the following:
- Whitespaces before and after
- hashtags
- Punctuations and emojis
- Recruiter Codes at the start of entry
- Any other recruiter codes within the text line or salary

In [8]:
REGEX_RULES = [
    ('recruiter_code', r'^\s*\d{3,6}\s*[-:]\s*', ' '),          # recruiter codes at the start
    ('hashtags', r'#\w+', ' '),                                 # hashtags
    ('other_numbers', r'\b(?=[a-z0-9]*\d)[a-z0-9]{3,}\b', ' '), # any other numbers of 3 and above
    ('punction_emojis', r'[^a-z]', ' '),                        # punctuations, emojis
    ('collapse_ws', r'\s+', ' '),                               # collapse whitespace
]

lower_title = df_clean['title'].str.lower()                     # make all cell values lowercase

for name, pattern, replace in REGEX_RULES:                      # reassignment with regex, replace if pattern matches
    lower_title = lower_title.str.replace(pattern, replace, regex=True)

df_clean['title_clean'] = lower_title.str.strip()               # keep 'title' column, create new column 'title_clean' with normalized entry


# after cleaning, some rows become empty. if so, replace with original 'title'

df_clean['title_clean'] = df_clean['title_clean'].where(
    df_clean['title_clean'] != '', df_clean['title'].str.lower())

## 1.4 Removing duplicated postings

Removing any duplicated postings due to original postings expiring. 4 parameters elected to identify duplicate rows:
- Using cleaned titles: 'title_clean'
- Using company name: 'postedCompany_name'
- Using salary: 'salary_minimum' and 'salary_maximum'

In [9]:
# Duplicated postings, defined as new postings of similar job due to previous posting expiring.

key_parameters = ['title_clean', 'postedCompany_name', 'salary_minimum', 'salary_maximum']

# Sort by original posting date first, drop duplicates but keep first posting
df_clean = df_clean.sort_values('metadata_originalPostingDate').drop_duplicates(subset= key_parameters, keep='first')

print(df_clean.shape)


(629246, 22)


## 1.5 Validating Position Levels against Years of Experience

To clean for positions with contradicting position levels vs years of experience

In [10]:
# Contradictions between stated job level and required experience.
JUNIOR_LEVELS = ['Fresh/entry level', 'Junior Executive']
SENIOR_LEVELS = ['Middle Management', 'Senior Management', 'Senior Executive']

years = df_clean['minimumYearsExperience']

# Senior label but almost no experience required
too_junior_for_label = df_clean['positionLevels'].isin(SENIOR_LEVELS) & (years < 3)

# Junior label but decades of experience required
too_senior_for_label = df_clean['positionLevels'].isin(JUNIOR_LEVELS) & (years > 20)

# Implausible experience regardless of label
implausible_years = years > 40

contradictory = too_junior_for_label | too_senior_for_label

print(f'Senior label, under 3 years:  {too_junior_for_label.sum():,}')
print(f'Junior label, over 20 years:  {too_senior_for_label.sum():,}')
print(f'Experience over 40 years:     {implausible_years.sum():,}')
print(f'Total contradictory labels:   {contradictory.sum():,} '
      f'({contradictory.mean():.2%} of postings)')

# Keep the row, blank the untrustworthy value. If we drop rows, it will affect hiring pool depth.
df_clean['positionLevels_clean'] = df_clean['positionLevels'].where(~contradictory)
df_clean.loc[implausible_years, 'minimumYearsExperience'] = np.nan

print(f"\npositionLevels_clean is null for "
      f"{df_clean['positionLevels_clean'].isna().sum():,} postings "
      f"({df_clean['positionLevels_clean'].isna().mean():.1%})")
df_clean['positionLevels_clean'].value_counts(dropna=False)

Senior label, under 3 years:  16,408
Junior label, over 20 years:  28
Experience over 40 years:     19
Total contradictory labels:   16,436 (2.61% of postings)

positionLevels_clean is null for 16,436 postings (2.6%)


positionLevels_clean
Executive            152290
Junior Executive      94169
Non-executive         81278
Professional          73458
Manager               72111
Fresh/entry level     58517
Senior Executive      48582
NaN                   16436
Middle Management     16372
Senior Management     16033
Name: count, dtype: int64

## 1.6 Dealing with Salary Outliers

Setting a boundary for the monthly salary to avoid outliers, not considering the typos for hourly rates and annual rates.

In [11]:
sal = ['salary_minimum', 'salary_maximum']

SALARY_FLOOR, SALARY_CEILING = 500, 60000

# Replace any empty salary with NaN
df_clean[sal] = df_clean[sal].replace(0, np.nan)

# Rebuild without 0 salary job postings
df_clean['average_salary'] = df_clean[sal].mean(axis=1)

# Set monthly pay range to remove any typos or wrong entries
df_clean_salary = df_clean[df_clean['average_salary'].between(SALARY_FLOOR, SALARY_CEILING)].copy()

## 1.6.1 Converting hourly and annual salaries to Monthly
We want to take into account typos for hour and annual salaries to be viable data to keep integrity of the data.

#### Rules:

| Rule | Reading | Action |
|---|---|---|
| `salary_min_clean` < 100 | hourly rate | x hours x 52 / 12 (44.3h full-time, 21h part-time, MoM averages) |
| min < 0.2 x max, max > 20,500 **AND** max/12 >= min | max quoted annually | max / 12 |
| min < 0.2 x max, max <= 20,500 | range too wide to trust | both set to the raw average |
| min > 40,000 | both quoted annually | both / 12 |



In [13]:
#####SETUP####
# Work from the raw bounds; only rows with a present value get touched.
df_clean['salary_min_clean'] = df_clean['salary_minimum']
df_clean['salary_max_clean'] = df_clean['salary_maximum']

has_min = df_clean['salary_min_clean'].notna()
has_max = df_clean['salary_max_clean'].notna()
both = has_min & has_max

# For rows with unreliable small maximum or extremely high maximum: unusable, blank it rather than drop the posting
df_clean.loc[has_max & ((df_clean['salary_max_clean'] < 5) | (df_clean['salary_max_clean'] >= 1000000)),
             ['salary_min_clean', 'salary_max_clean']] = np.nan
has_min = df_clean['salary_min_clean'].notna()
has_max = df_clean['salary_max_clean'].notna()
both = has_min & has_max

# For rows if only the maximum was filled in properly: mirror it into the minimum
only_max_ok = both & (df_clean['salary_min_clean'] < 1000) & \
              (df_clean['salary_min_clean'] < 0.2 * df_clean['salary_max_clean'])
df_clean.loc[only_max_ok, 'salary_min_clean'] = df_clean.loc[only_max_ok, 'salary_max_clean']
print(f'Minimum mirrored from maximum: {only_max_ok.sum():,}')

Minimum mirrored from maximum: 1,330


In [14]:
# Changing potential hourly wages to monthly
# Monthly = hourly x weekly hours x 52 / 12.  
# MoM average hours: 44.3 full-time, 21 part-time. https://stats.mom.gov.sg

hourly = df_clean['salary_min_clean'].notna() & (df_clean['salary_min_clean'] < 100)

# A senior role quoted at an hourly-looking rate is a data error, not an
# hourly job. Flag rather than convert.
salary_level_mismatch = hourly & df_clean['positionLevels_clean'].isin(SENIOR_LEVELS)
hourly = hourly & ~salary_level_mismatch

part_time = df_clean['employmentTypes'].eq('Part Time')
hours = np.where(part_time, 21, 44.3)
factor = pd.Series(hours * 52 / 12, index=df_clean.index)

for column in ['salary_min_clean', 'salary_max_clean']:
    df_clean.loc[hourly, column] = df_clean.loc[hourly, column] * factor[hourly]

print(f'Converted from hourly:        {hourly.sum():,}')
print(f'Senior roles at hourly rates: {salary_level_mismatch.sum():,} (flagged, not converted)')

Converted from hourly:        3,361
Senior roles at hourly rates: 39 (flagged, not converted)


In [15]:
# Changing potential annual wages to monthly

# A maximum is read as annual only if it is too large to be monthly AND
# dividing it by 12 leaves a figure still at or above the minimum. 

ANNUAL_MAX_THRESHOLD = 20500    # if higher than this, a maximum may be an annual figure
BOTH_ANNUAL_THRESHOLD = 40000   # a monthly minimum this high is rare apart from senior mgt roles

both = df_clean['salary_min_clean'].notna() & df_clean['salary_max_clean'].notna()
wide_range = both & (df_clean['salary_min_clean'] < 0.2 * df_clean['salary_max_clean'])
candidate_monthly = df_clean['salary_max_clean'] / 12

# BOTH masks are computed before either is applied. 
max_annual = (wide_range
              & (df_clean['salary_max_clean'] > ANNUAL_MAX_THRESHOLD)
              & (candidate_monthly >= df_clean['salary_min_clean']))

# A row that failed the test - too wide spread.
too_wide = wide_range & ~max_annual

# Maximum quoted annually, minimum monthly
df_clean.loc[max_annual, 'salary_max_clean'] = candidate_monthly[max_annual]

# If range too wide to - collapse both ends onto the raw average
df_clean.loc[too_wide, 'salary_min_clean'] = df_clean.loc[too_wide, 'average_salary']
df_clean.loc[too_wide, 'salary_max_clean'] = df_clean.loc[too_wide, 'average_salary']

# Both ends quoted annually. Evaluated after the steps above, so a minimum only
# revealed as annual by an earlier fix is still caught.
both_annual = df_clean['salary_min_clean'].notna() & \
              (df_clean['salary_min_clean'] > BOTH_ANNUAL_THRESHOLD)
for column in ['salary_min_clean', 'salary_max_clean']:
    df_clean.loc[both_annual, column] = df_clean.loc[both_annual, column] / 12

# Junior label on a very high monthly salary is a mismatch, not a conversion
junior_high = df_clean['positionLevels_clean'].isin(JUNIOR_LEVELS) & \
              (df_clean['salary_max_clean'] >= ANNUAL_MAX_THRESHOLD)
salary_level_mismatch = salary_level_mismatch | junior_high

assert not (max_annual & too_wide).any(), 'annual rules overlap - check the masks'

# None of the three rules can invert a range: the annual reading is refused
# unless it stays above the minimum, collapsing sets both ends equal, and
# dividing both ends preserves their order. This asserts that stays true.
inverted = both & (df_clean['salary_max_clean'] < df_clean['salary_min_clean'])
assert inverted.sum() == 0, (
    f'{inverted.sum():,} postings ended up with a maximum below their minimum')

print(f'Maximum converted from annual: {max_annual.sum():,}')
print(f'Range collapsed to average:    {too_wide.sum():,}')
print(f'Both ends from annual:         {both_annual.sum():,}')
print(f'Junior label, high pay:        {junior_high.sum():,} (flagged)')
print('No inverted ranges.')

Maximum converted from annual: 148
Range collapsed to average:    287
Both ends from annual:         302
Junior label, high pay:        46 (flagged)
No inverted ranges.


In [16]:
# Final clean

# converted average, and the mismatch flag the dashboard filters
df_clean['salary_min_clean'] = df_clean['salary_min_clean'].round()
df_clean['salary_max_clean'] = df_clean['salary_max_clean'].round()
df_clean['average_salary_clean'] = (
    df_clean[['salary_min_clean', 'salary_max_clean']].mean(axis=1)
)
df_clean['salary_level_mismatch'] = salary_level_mismatch

rescued = (df_clean['average_salary_clean'].between(SALARY_FLOOR, SALARY_CEILING)
           & ~df_clean['average_salary'].between(SALARY_FLOOR, SALARY_CEILING))
print(f'Postings rescued into the usable band by conversion: {rescued.sum():,}')
print()
print(df_clean[['average_salary', 'average_salary_clean']].describe().round(0))

Postings rescued into the usable band by conversion: 3,705

       average_salary  average_salary_clean
count        629246.0              627710.0
mean           5127.0                4966.0
std           32822.0                3292.0
min               1.0                   4.0
25%            2950.0                2975.0
50%            3950.0                4000.0
75%            6000.0                6000.0
max        12666400.0               65000.0


## 1.7 Parse Categories + id into list columns
We parse the categories into list columns on df_clean, then use .explode() to build the long table.

In [17]:
# Parse safely: return [] instead of raising, so one bad row cannot kill the run.

def parse_categories(json_string):
    if pd.isna(json_string):                                    # if empty, return [] instead of None
        return []
    try:
        items = json.loads(json_string)
    except (TypeError, ValueError):                             # if json.loads fail, return [] instead of Error
        return []
    if not isinstance(items, list):                             # if not list, return []
        return []
    # de-duplicate within a posting, keep original order
    seen, out = set(), []
    for c in items:
        if not isinstance(c, dict) or c.get('category') is None:
            continue
        name = str(c['category']).strip()
        if name in seen:
            continue
        seen.add(name)
        out.append((c.get('id'), name))
    return out


parsed = df_clean['categories'].apply(parse_categories)

# Two aligned list columns: same length per row, so they explode together later
df_clean['category_id_list'] = parsed.apply(lambda rows: [i for i, _ in rows])
df_clean['category_list']    = parsed.apply(lambda rows: [n for _, n in rows])
df_clean['n_categories']     = df_clean['category_list'].str.len()

# First category only for a quick groupby 
# User-facing filter must use category_list, not this.
df_clean['main_category'] = df_clean['category_list'].str[0]

print('Rows that failed to parse or were empty:', (df_clean['n_categories'] == 0).sum())
print()
print(df_clean['n_categories'].value_counts().sort_index())

Rows that failed to parse or were empty: 0

n_categories
1    417583
2    111329
3     52309
4     22422
5     25603
Name: count, dtype: int64


In [18]:
print(f"Postings in more than one industry: {(df_clean['n_categories'] > 1).mean():.1%}")
print(f"Postings with no industry at all:   {(df_clean['n_categories'] == 0).mean():.1%}")
print(f"Total postings:                     {len(df_clean):,}")
print(f"Total posting x category pairs:     {df_clean['n_categories'].sum():,}")

df_clean[['title', 'category_list', 'main_category', 'n_categories']].head(5)

Postings in more than one industry: 33.6%
Postings with no industry at all:   0.0%
Total postings:                     629,246
Total posting x category pairs:     1,014,871


,title,category_list,main_category,n_categories
15438,Quantity Surveyor - Structural Steel,"[Building and Construction, Engineering]",Building and Construction,2
20535,Preschool Teacher (Foreigner / Local),[Education and Training],Education and Training,1
23658,Haircut Specialist,"[Customer Service, Personal Care / Beauty, Sal...",Customer Service,4
23805,Senior Purchasing Executive (Marine),[Purchasing / Merchandising],Purchasing / Merchandising,1
17692,Assistant Chef,"[F&B, General Work]",F&B,2


## 1.8 .explode() into long table

In [19]:
df_categories_exploded = (
    df_clean[['metadata_jobPostId', 'category_id_list', 'category_list']]
    .explode(['category_id_list', 'category_list'])          # pass lists to explode, unnest in parallel
    .rename(columns={'category_id_list': 'category_id',
                     'category_list': 'category_name'})
    .dropna(subset=['category_name'])                        # drop the empty-category rows i.e. any rows with no categories
    .reset_index(drop=True)
)

# Ensuring same dtypes
df_categories_exploded['category_id'] = df_categories_exploded['category_id'].astype('int64')
df_categories_exploded['category_name'] = df_categories_exploded['category_name'].astype('category')

print(f"df_clean:                {len(df_clean):,} rows (one per posting)")
print(f"df_categories_exploded:  {len(df_categories_exploded):,} rows (one per posting x category)")
df_categories_exploded.head(8)

# Assign df_categories to exploded
df_categories = df_categories_exploded

df_clean:                629,246 rows (one per posting)
df_categories_exploded:  1,014,871 rows (one per posting x category)


## 1.8.1 Overview of all categories and ids

In [20]:
category_lookup = (
    df_categories[['category_id', 'category_name']]
    .drop_duplicates()
    .sort_values('category_id')
    .reset_index(drop=True)
)
print(f'{len(category_lookup)} distinct industries')

category_counts = df_categories['category_name'].value_counts()                     # count for each categories
category_lookup['postings'] = category_lookup['category_name'].map(category_counts) # map count using category name, to new column, postings
category_lookup

43 distinct industries


,category_id,category_name,postings
0,1,Accounting / Auditing / Taxation,49027
1,2,Admin / Secretarial,70357
2,3,Advertising / Media,11947
3,4,Architecture / Interior Design,9744
4,5,Banking and Finance,40026
5,6,Building and Construction,53918
6,7,Consulting,21389
7,8,Customer Service,62119
8,9,Design,12880
9,10,Education and Training,25157


## 1.9 Job Roles from 'title_clean'
Our brief covers benchmarking "by industry" and also "by role". 
This section covers the role dimension, using 'title_clean'

In [21]:
ROLE_PATTERNS = [
    ('Data / Analytics',           r'\b(data scientist|data analyst|data engineer|business intelligence|analytics|machine learning)\b'),
    ('Software Engineering',       r'\b(software engineer|developer|programmer|full stack|front end|back end|devops|qa engineer|test engineer)\b'),
    ('IT / Infrastructure',        r'\b(it (support|executive|manager|specialist)|system(s)? (admin|engineer|analyst)|network engineer|cyber ?security|cloud engineer|helpdesk|technical support)\b'),
    ('Product / Project',          r'\b(product manager|product owner|project manager|project executive|scrum master|business analyst)\b'),
    ('Design',                     r'\b(designer|creative director|art director|graphic)\b'),
    ('Engineering (Non-Software)', r'\b(mechanical|electrical|civil|structural|process|chemical|industrial|maintenance) engineer\b'),
    ('Sales / Business Dev',       r'\b(sales|business development|account (manager|executive)|relationship manager|retail assistant|promoter)\b'),
    ('Marketing / Comms',          r'\b(marketing|brand|content|social media|communications|public relations|copywriter)\b'),
    ('Finance / Accounting',       r'\b(account(s|ant|ing)|audit|tax|finance|financial|treasury|credit|bookkeep|payroll)\b'),
    ('Human Resources',            r'\b(hr|human resource|recruit|talent acquisition|people operations)\b'),
    ('Operations / Logistics',     r'\b(operations|logistics|warehouse|supply chain|procurement|purchasing|inventory|dispatch|driver|forklift)\b'),
    ('Healthcare',                 r'\b(nurse|nursing|doctor|physician|pharmacist|therapist|clinic|medical|dental|healthcare|caregiver)\b'),
    ('Education',                  r'\b(teacher|tutor|lecturer|trainer|educator|instructor|curriculum|childcare|preschool)\b'),
    ('Customer Service',           r'\b(customer service|customer support|call cent|service crew|receptionist|concierge|guest service)\b'),
    ('Food & Beverage',            r'\b(chef|cook|barista|waiter|waitress|kitchen|f b|restaurant|bartender|pastry)\b'),
    ('Legal / Compliance',         r'\b(legal|lawyer|solicitor|paralegal|compliance|regulatory|counsel)\b'),
    ('Admin / Secretarial',        r'\b(admin|administrative|secretar|clerk|clerical|data entry|personal assistant)\b'),
    ('Construction / Trades',      r'\b(construction|site (supervisor|engineer|manager)|foreman|carpenter|plumber|electrician|welder|painter|technician)\b'),
    ('Security / Facilities',      r'\b(security (officer|guard|supervisor)|cleaner|cleaning|housekeep|facilit|janitor)\b'),
]

ROLE_REGEX = [(label, re.compile(pattern)) for label, pattern in ROLE_PATTERNS]


def match_roles(title):                      # every role family the title matches
    if not isinstance(title, str) or not title:
        return []
    return [label for label, regex in ROLE_REGEX if regex.search(title)]


titles = df_clean['title_clean'].fillna('')
df_clean['role_list']    = titles.apply(match_roles)
df_clean['primary_role'] = df_clean['role_list'].apply(
    lambda r: r[0] if r else 'Other / Unclassified')

df_clean[['title', 'title_clean', 'primary_role', 'role_list']].head(10)

,title,title_clean,primary_role,role_list
15438,Quantity Surveyor - Structural Steel,quantity surveyor structural steel,Other / Unclassified,[]
20535,Preschool Teacher (Foreigner / Local),preschool teacher foreigner local,Education,[Education]
23658,Haircut Specialist,haircut specialist,Other / Unclassified,[]
23805,Senior Purchasing Executive (Marine),senior purchasing executive marine,Operations / Logistics,[Operations / Logistics]
17692,Assistant Chef,assistant chef,Food & Beverage,[Food & Beverage]
16957,Kitchen Crew,kitchen crew,Food & Beverage,[Food & Beverage]
18642,Assistant Chef,assistant chef,Food & Beverage,[Food & Beverage]
18694,Project Manager,project manager,Product / Project,[Product / Project]
16712,Service Crew,service crew,Customer Service,[Customer Service]
18762,Kitchen Crew,kitchen crew,Food & Beverage,[Food & Beverage]


## Analysis Grouping

In [22]:
# Salary usability

df_clean['salary_reliable'] = (
    df_clean['average_salary_clean'].between(SALARY_FLOOR, SALARY_CEILING)
    & ~df_clean['salary_level_mismatch']
)

print(f"Usable salary rows (converted): {df_clean['salary_reliable'].sum():,} "
      f"({df_clean['salary_reliable'].mean():.1%})")
print(f"df_clean_salary rows (raw):     {len(df_clean_salary):,}")
print(f"Net gain from conversion:       "
      f"{df_clean['salary_reliable'].sum() - len(df_clean_salary):+,}")

# The flag must be a superset of the raw filter, minus the level mismatches.
raw_ok = df_clean['average_salary'].between(SALARY_FLOOR, SALARY_CEILING)
lost = raw_ok & ~df_clean['salary_reliable'] & ~df_clean['salary_level_mismatch']
assert lost.sum() == 0, (
    f'{lost.sum():,} postings were usable before conversion and are not now - '
    'check the thresholds in 1.5.2')
print('\nNo posting was lost by the conversion.')

# Running describe on salary_reliable to confirm no outliers
print(df_clean.loc[df_clean['salary_reliable'],"average_salary_clean"].describe().round(0))

Usable salary rows (converted): 626,968 (99.6%)
df_clean_salary rows (raw):     623,309
Net gain from conversion:       +3,659

No posting was lost by the conversion.
count    626968.0
mean       4969.0
std        3283.0
min         500.0
25%        3000.0
50%        4000.0
75%        6000.0
max       55500.0
Name: average_salary_clean, dtype: float64


In [23]:
# Experience bands
def group_experience(years):
    if pd.isna(years):
        return None
    if years <= 1:
        return '0-1 years'
    if years <= 4:
        return '2-4 years'
    if years <= 9:
        return '5-9 years'
    return '10+ years'


# Employment type, grouped into standard vs flexible
EMPLOYMENT_GROUPS = {
    'Permanent': 'Standard', 'Full Time': 'Standard', 'Contract': 'Standard',
    'Part Time': 'Flexible / Non-Standard', 'Temporary': 'Flexible / Non-Standard',
    'Freelance': 'Flexible / Non-Standard', 'Flexi-work': 'Flexible / Non-Standard',
    'Internship/Attachment': 'Internship',
}

df_clean['experience_group'] = df_clean['minimumYearsExperience'].apply(group_experience)
df_clean['employment_group'] = df_clean['employmentTypes'].map(EMPLOYMENT_GROUPS).fillna('Other')
df_clean['posting_month']    = df_clean['metadata_originalPostingDate'].dt.to_period('M').dt.to_timestamp()

missing_exp = df_clean['experience_group'].isna().mean()
print(f'Postings with no experience value: {missing_exp:.1%} '
      '(excluded from any experience-band filter)\n')

for column in ['experience_group', 'employment_group']:
    print(f'--- {column} ---')
    print(df_clean[column].value_counts(dropna=False), end='\n\n')

Postings with no experience value: 0.0% (excluded from any experience-band filter)

--- experience_group ---
experience_group
2-4 years    264010
0-1 years    208858
5-9 years    128556
10+ years     27803
NaN              19
Name: count, dtype: int64

--- employment_group ---
employment_group
Standard                   598650
Flexible / Non-Standard     26472
Internship                   4124
Name: count, dtype: int64



## Writing Files
Output for 2 csv files
- SGJobData_clean.csv
- SGJobData_categories.csv : this is the exploded table for industry breakdown

In [62]:
SEP = '|'   # csv cannot store python lists, 3 list columns joined with |

df_out = df_clean.copy()
for column in ['category_list', 'category_id_list', 'role_list']:
    df_out[column] = df_out[column].apply(
        lambda items: SEP.join(str(i) for i in items) if len(items) else '')

df_out = df_out.drop(columns=['categories'], errors='ignore')

df_out.to_csv('SGJobData_clean.csv', index=False)
df_categories.to_csv('SGJobData_categories.csv', index=False)


## Streamlit Dashboard
Overview: Client picks a target industry or role and gets prevailing salary ranges and a read on hiring pool.

In [ ]:
%%writefile app.py
"""
Singapore hiring benchmark for market entry

Purpose
-------
Help companies estimate:
1. Where hiring demand is concentrated
2. What salary budget they should plan for
3. Which sectors appear tighter to hire for

Run
---
    python -m streamlit run app.py
"""

from pathlib import Path

import altair as alt
import pandas as pd
import streamlit as st


HERE = Path(__file__).resolve().parent
CLEAN_PATH = HERE / "SGJobData_clean.csv"
SALARY_COL = "average_salary_clean"
DATE_COLS = [
    "metadata_expiryDate",
    "metadata_newPostingDate",
    "metadata_originalPostingDate",
    "posting_month",
]

SALARY_FLOOR = 500
SALARY_CEILING = 60_000
EXPERIENCE_ORDER = ["0-1 years", "2-4 years", "5-9 years", "10+ years"]


st.set_page_config(
    page_title="Singapore hiring benchmark",
    page_icon=":material/query_stats:",
    layout="wide",
    initial_sidebar_state="expanded",
)


def group_experience(years):
    if pd.isna(years):
        return None
    if years <= 1:
        return "0-1 years"
    if years <= 4:
        return "2-4 years"
    if years <= 9:
        return "5-9 years"
    return "10+ years"


def normalize_bool(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.lower().isin(["true", "1", "yes"])


@st.cache_data(show_spinner="Loading cleaned jobs data...")
def load_data():
    try:
        df = pd.read_csv(CLEAN_PATH, parse_dates=DATE_COLS)
    except ValueError:
        df = pd.read_csv(
            CLEAN_PATH,
            parse_dates=[
                "metadata_expiryDate",
                "metadata_newPostingDate",
                "metadata_originalPostingDate",
            ],
        )

    numeric_cols = [
        "metadata_totalNumberJobApplication",
        "numberOfVacancies",
        "metadata_totalNumberOfView",
        "metadata_repostCount",
        "salary_minimum",
        "salary_maximum",
        "average_salary",
        "average_salary_clean",
        "minimumYearsExperience",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if SALARY_COL not in df.columns and "average_salary" in df.columns:
        df[SALARY_COL] = df["average_salary"]

    if "salary_reliable" in df.columns:
        df["salary_reliable"] = normalize_bool(df["salary_reliable"])
    else:
        df["salary_reliable"] = df[SALARY_COL].between(SALARY_FLOOR, SALARY_CEILING)

    if "salary_min_clean" not in df.columns and "salary_minimum" in df.columns:
        df["salary_min_clean"] = df["salary_minimum"]
    if "salary_max_clean" not in df.columns and "salary_maximum" in df.columns:
        df["salary_max_clean"] = df["salary_maximum"]

    if "posting_month" not in df.columns:
        df["posting_month"] = (
            pd.to_datetime(df["metadata_originalPostingDate"], errors="coerce")
            .dt.to_period("M")
            .dt.to_timestamp()
        )
    else:
        df["posting_month"] = pd.to_datetime(df["posting_month"], errors="coerce")

    if "experience_group" not in df.columns and "minimumYearsExperience" in df.columns:
        df["experience_group"] = df["minimumYearsExperience"].apply(group_experience)

    df["main_category"] = df["main_category"].fillna("Unknown")
    df["primary_role"] = df["primary_role"].fillna("Other / Unclassified")
    df["title_clean"] = (
        df["title_clean"]
        if "title_clean" in df.columns
        else df["title"].astype(str).str.strip().str.lower()
    )

    vacancies = df["numberOfVacancies"].where(df["numberOfVacancies"] > 0)
    views = df["metadata_totalNumberOfView"].where(df["metadata_totalNumberOfView"] > 0)

    # Candidate-response metrics. These are more informative than repost counts
    # for this dataset because repost values are overwhelmingly zero-heavy.
    df["applications_per_vacancy"] = df["metadata_totalNumberJobApplication"] / vacancies
    df["application_rate"] = df["metadata_totalNumberJobApplication"] / views
    df["role_classified"] = df["primary_role"].ne("Other / Unclassified")

    return df


def apply_filters(frame, filters):
    dff = frame.copy()

    if filters["categories"]:
        dff = dff[dff["main_category"].isin(filters["categories"])]
    if filters["roles"]:
        dff = dff[dff["primary_role"].isin(filters["roles"])]
    if filters["positions"]:
        dff = dff[dff["positionLevels"].isin(filters["positions"])]
    if filters["employment"]:
        dff = dff[dff["employmentTypes"].isin(filters["employment"])]
    if filters["experience"]:
        dff = dff[dff["experience_group"].isin(filters["experience"])]
    if filters["title_query"]:
        dff = dff[
            dff["title_clean"].str.contains(filters["title_query"], case=False, na=False)
        ]

    start = pd.Timestamp(filters["date_range"][0])
    end = pd.Timestamp(filters["date_range"][1]) + pd.Timedelta(days=1)
    dff = dff[dff["metadata_originalPostingDate"].between(start, end)]
    return dff


def sector_benchmarks(frame, salary_frame, min_postings):
    base = (
        frame.groupby("main_category")
        .agg(
            postings=("metadata_jobPostId", "nunique"),
            mean_applications_per_vacancy=("applications_per_vacancy", "mean"),
            mean_application_rate=("application_rate", "mean"),
        )
        .reset_index()
    )

    salary = (
        salary_frame
        .groupby("main_category")[SALARY_COL]
        .median()
        .rename("median_salary")
        .reset_index()
    )

    bench = base.merge(salary, on="main_category", how="left")
    bench = bench[bench["postings"] >= min_postings].copy()
    bench = bench.dropna(
        subset=["mean_applications_per_vacancy", "mean_application_rate", "median_salary"]
    )

    if bench.empty:
        return bench

    # Higher score = tighter market for employers:
    # lower candidate response + higher pay requirement.
    bench["tightness_score"] = (
        (1 - bench["mean_applications_per_vacancy"].rank(pct=True)) * 0.45
        + (1 - bench["mean_application_rate"].rank(pct=True)) * 0.35
        + bench["median_salary"].rank(pct=True) * 0.20
    )

    bench["tightness_label"] = pd.cut(
        bench["tightness_score"],
        bins=[-0.01, 0.25, 0.50, 0.75, 1.01],
        labels=["Easier", "Balanced", "Tighter", "Tightest"],
    )
    return bench.sort_values("tightness_score", ascending=False)


def role_response_benchmarks(frame, min_postings):
    role = (
        frame[frame["role_classified"]]
        .groupby("primary_role")
        .agg(
            postings=("metadata_jobPostId", "nunique"),
            mean_applications_per_vacancy=("applications_per_vacancy", "mean"),
            mean_application_rate=("application_rate", "mean"),
        )
        .reset_index()
    )
    role = role[role["postings"] >= min_postings].copy()
    return role.sort_values("mean_applications_per_vacancy", ascending=True)


def money(value):
    if pd.isna(value):
        return "N/A"
    return f"S${value:,.0f}"


def top_highlight_chart(data, category_col, value_col, highlight_value, x_title, height=360):
    chart_data = data.copy()
    chart_data["highlight_group"] = chart_data[category_col].apply(
        lambda x: "Highlighted" if x == highlight_value else "Other"
    )
    return (
        alt.Chart(chart_data)
        .mark_bar()
        .encode(
            x=alt.X(f"{value_col}:Q", title=x_title),
            y=alt.Y(f"{category_col}:N", sort="-x", title=None),
            color=alt.Color(
                "highlight_group:N",
                scale=alt.Scale(
                    domain=["Highlighted", "Other"],
                    range=["#0f766e", "#cbd5e1"],
                ),
                legend=None,
            ),
            tooltip=[category_col, value_col],
        )
        .properties(height=height)
    )


def salary_by_experience(frame):
    result = (
        frame[frame["salary_reliable"]]
        .groupby("experience_group")[SALARY_COL]
        .agg(median_salary="median", postings="size")
        .reset_index()
    )
    result["experience_group"] = pd.Categorical(
        result["experience_group"], categories=EXPERIENCE_ORDER, ordered=True
    )
    return result.sort_values("experience_group")


def salary_trend_by_experience(frame):
    result = (
        frame[frame["salary_reliable"]]
        .groupby(["posting_month", "experience_group"])[SALARY_COL]
        .median()
        .reset_index(name="median_salary")
    )
    result["experience_group"] = pd.Categorical(
        result["experience_group"], categories=EXPERIENCE_ORDER, ordered=True
    )
    return result.sort_values(["experience_group", "posting_month"])


df = load_data()

min_date = df["metadata_originalPostingDate"].min().date()
max_date = df["metadata_originalPostingDate"].max().date()

category_options = sorted(df["main_category"].dropna().unique().tolist())
role_options = sorted(
    df.loc[df["role_classified"], "primary_role"].dropna().unique().tolist()
)
position_options = sorted(df["positionLevels"].dropna().unique().tolist())
employment_options = sorted(df["employmentTypes"].dropna().unique().tolist())

with st.sidebar:
    st.header("Filters")

    if st.button("Reset filters"):
        st.session_state.clear()
        st.rerun()

    selected_dates = st.date_input(
        "Posting date range",
        value=(min_date, max_date),
        min_value=min_date,
        max_value=max_date,
    )

    selected_categories = st.multiselect("Sector", category_options)
    selected_roles = st.multiselect("Role family", role_options)
    selected_positions = st.multiselect("Position level", position_options)
    selected_employment = st.multiselect("Employment type", employment_options)
    selected_experience = st.multiselect("Experience band", EXPERIENCE_ORDER)
    title_query = st.text_input("Title contains", placeholder="engineer, analyst, nurse")

    salary_only = st.checkbox(
        "Use salary-clean rows only",
        value=True,
        help=f"Limits salary analysis to rows between S${SALARY_FLOOR:,} and S${SALARY_CEILING:,}.",
    )
    min_n = st.slider(
        "Minimum postings per benchmark group",
        min_value=25,
        max_value=1000,
        value=100,
        step=25,
    )

filters = {
    "date_range": selected_dates if len(selected_dates) == 2 else (min_date, max_date),
    "categories": selected_categories,
    "roles": selected_roles,
    "positions": selected_positions,
    "employment": selected_employment,
    "experience": selected_experience,
    "title_query": title_query.strip(),
    "salary_only": salary_only,
}

filtered_df = apply_filters(df, filters)
salary_frame = (
    filtered_df[filtered_df["salary_reliable"]].copy()
    if salary_only
    else filtered_df[filtered_df[SALARY_COL].notna()].copy()
)
market_salary_frame = (
    df[df["salary_reliable"]].copy()
    if salary_only
    else df[df[SALARY_COL].notna()].copy()
)

market_bench = sector_benchmarks(filtered_df, salary_frame, min_n)
role_bench = role_response_benchmarks(filtered_df, min_n)
salary_exp = salary_by_experience(salary_frame)
salary_trend = salary_trend_by_experience(salary_frame)

st.title("Singapore hiring benchmark for market entry")
st.caption(
    "Use this view to judge where hiring demand sits, what salary budget to plan, "
    "and which sectors look tighter for employers trying to build a team in Singapore."
)

if filtered_df.empty:
    st.warning("No rows match the selected filters.")
    st.stop()

salary_rows = salary_frame
classified_share = filtered_df["role_classified"].mean()
top_sector = filtered_df["main_category"].mode().iloc[0]
median_salary = salary_rows[SALARY_COL].median() if not salary_rows.empty else float("nan")
market_salary = market_salary_frame[SALARY_COL].median() if not market_salary_frame.empty else float("nan")
mean_apv = filtered_df["applications_per_vacancy"].mean()
market_apv = df["applications_per_vacancy"].mean()
mean_rate = filtered_df["application_rate"].mean()
market_rate = df["application_rate"].mean()
salary_label = f"S${median_salary:,.0f}" if pd.notna(median_salary) else "N/A"

coverage_monthly = (
    filtered_df.groupby("posting_month")["metadata_jobPostId"]
    .nunique()
    .reset_index(name="postings")
    .sort_values("posting_month")
)
coverage_cutoff_date = None
if not coverage_monthly.empty:
    coverage_threshold = coverage_monthly["postings"].max() * 0.25
    coverage_match = coverage_monthly[coverage_monthly["postings"] >= coverage_threshold]
    if not coverage_match.empty:
        coverage_cutoff_date = coverage_match["posting_month"].min()

if mean_apv >= market_apv * 1.15 and mean_rate >= market_rate * 1.10:
    response_label = "stronger than the wider market"
elif mean_apv <= market_apv * 0.85 and mean_rate <= market_rate * 0.90:
    response_label = "weaker than the wider market"
else:
    response_label = "broadly in line with the wider market"

st.info(
    f"This filtered market is led by **{top_sector}**. "
    f"Median monthly pay is **{salary_label}**, "
    f"and candidate response is **{response_label}** with **{mean_apv:.2f} applications per vacancy** "
    f"and an average **{mean_rate:.1%} application rate**."
)

with st.container(horizontal=True):
    st.metric("Postings in view", f"{filtered_df['metadata_jobPostId'].nunique():,}", border=True)
    st.metric("Hiring companies", f"{filtered_df['postedCompany_name'].nunique():,}", border=True)
    st.metric(
        "Median monthly salary",
        salary_label,
        delta=f"{median_salary - market_salary:+,.0f} vs dataset"
        if pd.notna(median_salary) and pd.notna(market_salary)
        else None,
        border=True,
    )
    st.metric(
        "Avg applications per vacancy",
        f"{mean_apv:.2f}",
        delta=f"{mean_apv - market_apv:+.2f} vs dataset",
        border=True,
    )
    st.metric(
        "Avg application rate",
        f"{mean_rate:.1%}",
        delta=f"{mean_rate - market_rate:+.1%} vs dataset",
        border=True,
    )

with st.expander("How to read the hiring competition metrics", icon=":material/help:"):
    st.markdown(
        """
        - **Applications per vacancy** = total applications divided by the number of vacancies in each posting.
        - **Application rate** = total applications divided by total views.
        - In this dataset, **higher values mean stronger candidate response**, which usually suggests it is easier to attract applicants.
        - We do **not** use median repost count or days-open as the main hiring benchmark because repost data is too zero-heavy and posting duration is largely driven by platform rules.
        """
    )

tab1, tab2, tab3, tab4 = st.tabs(
    ["Where demand sits", "What to budget", "Where hiring looks tighter", "Filtered data"]
)

with tab1:
    c1, c2 = st.columns(2)

    demand_by_sector = (
        filtered_df.groupby("main_category")["metadata_jobPostId"]
        .nunique()
        .reset_index(name="postings")
        .sort_values("postings", ascending=False)
        .head(10)
    )

    with c1:
        if demand_by_sector.empty:
            st.subheader("Sector demand overview")
            st.caption("No sector demand view is available for the current filters.")
        else:
            demand_leader = demand_by_sector.iloc[0]
            st.subheader(
                f"{demand_leader['main_category']} has the strongest hiring demand in this view"
            )
            sector_chart = top_highlight_chart(
                demand_by_sector,
                "main_category",
                "postings",
                demand_leader["main_category"],
                "Unique postings",
            )
            st.altair_chart(sector_chart)
            st.caption(
                f"The highlighted bar shows the leading sector with **{demand_leader['postings']:,}** unique postings."
            )

    demand_over_time = (
        filtered_df.groupby("posting_month")["metadata_jobPostId"]
        .nunique()
        .reset_index(name="postings")
        .sort_values("posting_month")
    )

    with c2:
        st.subheader("Hiring volume becomes more reliable from March 2023 onward")
        trend_base = alt.Chart(demand_over_time).encode(
            x=alt.X("posting_month:T", title="Posting month"),
            y=alt.Y("postings:Q", title="Unique postings"),
            tooltip=["posting_month:T", "postings"],
        )
        trend_chart = trend_base.mark_line(point=True, color="#1d4ed8")
        layers = []
        if coverage_cutoff_date is not None:
            shaded = alt.Chart(
                pd.DataFrame(
                    {
                        "start": [demand_over_time["posting_month"].min()],
                        "end": [coverage_cutoff_date],
                    }
                )
            ).mark_rect(color="#e2e8f0", opacity=0.65).encode(
                x="start:T",
                x2="end:T",
            )
            layers.append(shaded)
        layers.append(trend_chart)
        st.altair_chart(alt.layer(*layers).properties(height=360))
        if coverage_cutoff_date is not None:
            st.caption(
                f"Months before **{coverage_cutoff_date:%B %Y}** are shaded because posting volume is much lower and likely reflects weaker dataset collection coverage."
            )
        else:
            st.caption(
                "Read this directionally. It shows the hiring volume captured by this dataset, not the full economy."
            )

    d1, d2 = st.columns(2)

    experience_mix = (
        filtered_df.groupby("experience_group")["metadata_jobPostId"]
        .nunique()
        .reindex(EXPERIENCE_ORDER)
        .reset_index(name="postings")
    )

    with d1:
        st.subheader("Experience mix in this market view")
        exp_chart = (
            alt.Chart(experience_mix.dropna())
            .mark_bar(color="#7c3aed")
            .encode(
                x=alt.X("experience_group:N", sort=EXPERIENCE_ORDER, title="Experience band"),
                y=alt.Y("postings:Q", title="Unique postings"),
                tooltip=["experience_group", "postings"],
            )
            .properties(height=320)
        )
        st.altair_chart(exp_chart)
        st.caption("This helps you see whether the market is leaning toward entry, mid-level, or senior hiring.")

    role_mix = (
        filtered_df[filtered_df["role_classified"]]
        .groupby("primary_role")["metadata_jobPostId"]
        .nunique()
        .reset_index(name="postings")
        .sort_values("postings", ascending=False)
        .head(10)
    )

    with d2:
        st.subheader("Top job families across those sector postings")
        if role_mix.empty:
            st.caption("No job-family view is available in the current filters.")
        else:
            role_leader = role_mix.iloc[0]
            role_chart = top_highlight_chart(
                role_mix,
                "primary_role",
                "postings",
                role_leader["primary_role"],
                "Unique postings",
                height=320,
            )
            st.altair_chart(role_chart)
            st.caption(
                f"Here, **sector** means the employer's industry, while **job family** means the type of role being hired, such as software, finance, or operations. "
                f"About **{classified_share:.0%}** of postings in this view have a usable job-family label."
            )

with tab2:
    s1, s2 = st.columns(2)

    sector_salary = (
        salary_rows.groupby("main_category")[SALARY_COL]
        .agg(median_salary="median", postings="size")
        .reset_index()
    )
    sector_salary = sector_salary[sector_salary["postings"] >= min_n].sort_values(
        "median_salary", ascending=False
    )

    with s1:
        if sector_salary.empty:
            st.subheader("Sector pay benchmark")
        else:
            salary_leader = sector_salary.iloc[0]
            st.subheader(
                f"{salary_leader['main_category']} has the highest median pay in this view"
            )
        if sector_salary.empty:
            st.caption("No sector has enough salary-clean rows for a stable benchmark.")
        else:
            top_salary_chart = sector_salary.head(10).copy()
            sal_chart = top_highlight_chart(
                top_salary_chart,
                "main_category",
                "median_salary",
                salary_leader["main_category"],
                "Median monthly salary (S$)",
            )
            st.altair_chart(sal_chart)
            st.caption(
                f"The highlighted bar shows the most expensive sector in this view at **{money(salary_leader['median_salary'])}** median monthly pay."
            )

    with s2:
        st.subheader("Median pay by experience band")
        if salary_exp.empty:
            st.caption("No salary-clean rows are available in this view.")
        else:
            exp_salary_chart = (
                alt.Chart(salary_exp.dropna())
                .mark_bar(color="#2563eb")
                .encode(
                    x=alt.X("experience_group:N", sort=EXPERIENCE_ORDER, title="Experience band"),
                    y=alt.Y("median_salary:Q", title="Median monthly salary (S$)"),
                    tooltip=["experience_group", "median_salary", "postings"],
                )
                .properties(height=360)
            )
            st.altair_chart(exp_salary_chart)
            st.caption("This is the clearest salary planning view for building a team tier by tier.")

    st.subheader("Salary trend by experience band")
    if salary_trend.empty:
        st.caption("No salary trend is available in this view.")
    else:
        trend_salary_chart = (
            alt.Chart(salary_trend.dropna())
            .mark_line(point=True)
            .encode(
                x=alt.X("posting_month:T", title="Posting month"),
                y=alt.Y("median_salary:Q", title="Median monthly salary (S$)"),
                color=alt.Color("experience_group:N", sort=EXPERIENCE_ORDER, title="Experience band"),
                tooltip=["posting_month:T", "experience_group", "median_salary"],
            )
            .properties(height=380)
        )
        st.altair_chart(trend_salary_chart)
        st.caption(
            "This segmented view is safer than a single overall salary trend because it reduces mix effects across seniority levels."
        )

with tab3:
    t1, t2 = st.columns(2)

    with t1:
        selected_sector_name = None
        selected_sector_row = None

        if market_bench.empty:
            st.subheader("Sector hiring map")
            st.caption("Not enough stable sector data is available for a hiring benchmark.")
        else:
            selected_sector_name = st.selectbox(
                "Highlight a sector on the map",
                market_bench["main_category"].tolist(),
                index=0,
            )
            selected_sector_row = market_bench[
                market_bench["main_category"] == selected_sector_name
            ].iloc[0]
            st.subheader(f"{selected_sector_name} is highlighted on the sector hiring map")

            x_rule = pd.DataFrame({"median_salary": [market_bench["median_salary"].median()]})
            y_rule = pd.DataFrame(
                {
                    "mean_applications_per_vacancy": [
                        market_bench["mean_applications_per_vacancy"].median()
                    ]
                }
            )

            chart_data = market_bench.copy()
            label_names = set(chart_data.head(4)["main_category"].tolist())
            label_names.add(selected_sector_name)
            chart_data["highlight_group"] = chart_data["main_category"].apply(
                lambda x: "Highlighted sector" if x == selected_sector_name else "Other sectors"
            )
            chart_data["label_text"] = chart_data["main_category"].apply(
                lambda x: x if x in label_names else None
            )

            points = (
                alt.Chart(chart_data)
                .mark_circle(size=180, opacity=0.85)
                .encode(
                    x=alt.X("median_salary:Q", title="Median monthly salary (S$)"),
                    y=alt.Y(
                        "mean_applications_per_vacancy:Q",
                        title="Average applications per vacancy",
                    ),
                    color=alt.Color(
                        "highlight_group:N",
                        scale=alt.Scale(
                            domain=["Highlighted sector", "Other sectors"],
                            range=["#c2410c", "#cbd5e1"],
                        ),
                        legend=None,
                    ),
                    size=alt.Size("postings:Q", title="Postings"),
                    tooltip=[
                        "main_category",
                        "postings",
                        "median_salary",
                        "mean_applications_per_vacancy",
                        "mean_application_rate",
                        "tightness_label",
                    ],
                )
            )

            labels = (
                alt.Chart(chart_data)
                .transform_filter("datum.label_text != null")
                .mark_text(align="left", dx=8, dy=-8, fontSize=11, color="#0f172a")
                .encode(
                    x="median_salary:Q",
                    y="mean_applications_per_vacancy:Q",
                    text="label_text:N",
                )
            )

            vline = alt.Chart(x_rule).mark_rule(strokeDash=[6, 4], color="gray").encode(
                x="median_salary:Q"
            )
            hline = alt.Chart(y_rule).mark_rule(strokeDash=[6, 4], color="gray").encode(
                y="mean_applications_per_vacancy:Q"
            )

            st.altair_chart((points + labels + vline + hline).properties(height=380))
            st.caption(
                f"{selected_sector_name} offers median pay of **{money(selected_sector_row['median_salary'])}** "
                f"and attracts **{selected_sector_row['mean_applications_per_vacancy']:.2f} applications per vacancy**. "
                "On this map, sectors that sit lower and further right are usually tougher for employers."
            )

    with t2:
        if market_bench.empty:
            st.subheader("Sector tightness ranking")
        else:
            tightest_sector = market_bench.iloc[0]
            st.subheader(
                f"{tightest_sector['main_category']} looks tightest for employers in this view"
            )
        if market_bench.empty:
            st.caption("No stable tightness benchmark is available.")
        else:
            st.dataframe(
                market_bench[
                    [
                        "main_category",
                        "postings",
                        "median_salary",
                        "mean_applications_per_vacancy",
                        "mean_application_rate",
                        "tightness_label",
                    ]
                ].head(12),
                hide_index=True,
                column_config={
                    "main_category": "Sector",
                    "postings": st.column_config.NumberColumn("Postings", format="%d"),
                    "median_salary": st.column_config.NumberColumn(
                        "Median monthly salary", format="S$ %.0f"
                    ),
                    "mean_applications_per_vacancy": st.column_config.NumberColumn(
                        "Avg apps / vacancy", format="%.2f"
                    ),
                    "mean_application_rate": st.column_config.NumberColumn(
                        "Avg application rate", format="percent"
                    ),
                    "tightness_label": "Hiring tightness",
                },
            )
            st.caption(
                f"The current tightest sector is **{tightest_sector['main_category']}** with "
                f"median pay of **{money(tightest_sector['median_salary'])}**, "
                f"**{tightest_sector['mean_applications_per_vacancy']:.2f} applications per vacancy**, "
                f"and an average application rate of **{tightest_sector['mean_application_rate']:.1%}**."
            )

    if role_bench.empty:
        st.subheader("Role-family response benchmark")
    else:
        weakest_role = role_bench.iloc[0]
        st.subheader(
            f"{weakest_role['primary_role']} gets the weakest applicant response among job families in this view"
        )
    if role_bench.empty:
        st.caption("No job-family benchmark is available for this view.")
    else:
        weakest_roles = role_bench.head(10).copy()
        weakest_roles["highlight_group"] = weakest_roles["primary_role"].apply(
            lambda x: "Weakest response" if x == weakest_role["primary_role"] else "Other roles"
        )
        weak_role_chart = (
            alt.Chart(weakest_roles)
            .mark_bar()
            .encode(
                x=alt.X(
                    "mean_applications_per_vacancy:Q",
                    title="Average applications per vacancy",
                ),
                y=alt.Y("primary_role:N", sort="x", title=None),
                color=alt.Color(
                    "highlight_group:N",
                    scale=alt.Scale(
                        domain=["Weakest response", "Other roles"],
                        range=["#b91c1c", "#cbd5e1"],
                    ),
                    legend=None,
                ),
                tooltip=[
                    "primary_role",
                    "postings",
                    "mean_applications_per_vacancy",
                    "mean_application_rate",
                ],
            )
            .properties(height=340)
        )
        st.altair_chart(weak_role_chart)
        st.caption(
            f"**{weakest_role['primary_role']}** attracts only **{weakest_role['mean_applications_per_vacancy']:.2f} applications per vacancy**, "
            f"which is well below the filtered-view average of **{mean_apv:.2f}**. "
            "This chart excludes `Other / Unclassified` so the comparison stays usable."
        )

with tab4:
    st.subheader("Filtered data preview")
    show_cols = [
        "title",
        "postedCompany_name",
        "main_category",
        "primary_role",
        "positionLevels",
        "employmentTypes",
        "minimumYearsExperience",
        "experience_group",
        "salary_min_clean",
        "salary_max_clean",
        SALARY_COL,
        "metadata_totalNumberJobApplication",
        "numberOfVacancies",
        "applications_per_vacancy",
        "metadata_totalNumberOfView",
        "application_rate",
        "metadata_originalPostingDate",
    ]
    show_cols = [col for col in show_cols if col in filtered_df.columns]

    st.dataframe(
        filtered_df[show_cols].head(1000),
        hide_index=True,
        column_config={
            "salary_min_clean": st.column_config.NumberColumn("Salary min", format="S$ %.0f"),
            "salary_max_clean": st.column_config.NumberColumn("Salary max", format="S$ %.0f"),
            SALARY_COL: st.column_config.NumberColumn("Average salary", format="S$ %.0f"),
            "applications_per_vacancy": st.column_config.NumberColumn(
                "Apps / vacancy", format="%.2f"
            ),
            "application_rate": st.column_config.NumberColumn(
                "Application rate", format="percent"
            ),
            "metadata_originalPostingDate": st.column_config.DateColumn(
                "Posting date", format="YYYY-MM-DD"
            ),
        },
    )
    st.caption("Showing the first 1,000 filtered rows.")

    st.download_button(
        "Download filtered rows as CSV",
        filtered_df[show_cols].to_csv(index=False).encode("utf-8"),
        file_name="filtered_jobs.csv",
        mime="text/csv",
    )
